In [5]:
from pathlib import Path
import importlib
import json
import sys
import pandas as pd

ROOT = Path.cwd()
FOS_ROOT = ROOT if ROOT.name == "freight_opportunity_score" else ROOT / "freight_opportunity_score"
SRC = FOS_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import freight_opportunity_score.fos_model as fos_model
importlib.reload(fos_model)
FreightOpportunityScorer = fos_model.FreightOpportunityScorer

DATA_PATH = FOS_ROOT / "fos_data" / "freight_opportunity_daily.csv"
ARTIFACT_DIR = FOS_ROOT / "artifacts"
REPORT_DIR = FOS_ROOT / "reports"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

data = pd.read_csv(DATA_PATH, low_memory=False)
data["date"] = pd.to_datetime(data["date"])

scorer = FreightOpportunityScorer(ARTIFACT_DIR)
fit_metadata = scorer.fit(data, train_end="2023-12-31")
scored = scorer.score(data, horizon=30)
backtest_predictions, metrics = scorer.backtest(scored, data)

scorer.save({
    "model_name": "RandomForestRegressor",
    "purpose": "Leakage-safe 30-day route freight return model used as FOS forecast fallback",
    "forecast_precedence": "Use model_forecast_* columns when non-null; otherwise use this temporal model",
    "training": fit_metadata,
    "component_weights": {
        "forecast": 0.25,
        "rate_opportunity": 0.15,
        "market_signal": 0.15,
        "fleet_supply": 0.10,
        "port_congestion": 0.10,
        "weather_risk": 0.10,
        "voyage_economics": 0.15,
    },
    "decision_thresholds": {
        "AVOID_WAIT": "fos < 20",
        "WAIT": "20 <= fos < 40",
        "MONITOR": "40 <= fos < 60",
        "GOOD_OPPORTUNITY": "60 <= fos < 80",
        "FIX_NOW": "fos >= 80",
    },
    "leakage_policy": "future_*, target, label, actual_after, and backtest columns excluded from model features",
})
scored.to_csv(REPORT_DIR / "fos_predictions.csv", index=False)
backtest_predictions.to_csv(REPORT_DIR / "fos_backtest_predictions.csv", index=False)
metrics.to_csv(REPORT_DIR / "fos_backtest_metrics.csv", index=False)
scorer.feature_importance().to_csv(REPORT_DIR / "fos_feature_importance.csv", index=False)
(REPORT_DIR / "fos_data_quality.json").write_text(json.dumps({
    "rows": len(data),
    "routes": int(data["route_id"].nunique()),
    "date_min": str(data["date"].min().date()),
    "date_max": str(data["date"].max().date()),
    "external_forecast_rows": int(data.get("forecast_model_available", pd.Series(0)).sum()),
    "future_columns_used_as_features": [c for c in scorer.model_features if c.startswith("future_")],
}, indent=2))

print(f"Rows scored: {len(scored):,}")
print(f"Features: {len(scorer.model_features)}")
print(f"Forecast source: {scored['forecast_source'].value_counts().to_dict()}")
print(scored["fos_recommendation"].value_counts().sort_index())
print(metrics.to_string(index=False))

Rows scored: 64,860
Features: 85
Forecast source: {'temporal_fos_return_model': 64860}
fos_recommendation
AVOID_WAIT              0
WAIT                 1336
MONITOR             47206
GOOD_OPPORTUNITY    16318
FIX_NOW                 0
Name: count, dtype: int64
 horizon_days  observations  hit_rate  precision_fix_now  false_signal_rate  savings_usd_mt
            7         64650  0.199134           0.714635           0.074277    17327.280158
           30         63960  0.267151           0.968527           0.008021    82773.189428
           60         63060  0.222645           0.823230           0.044640   103997.439452
